# Detección de SPAM con Machine Learning

Clasificador binario de correos electrónicos (spam / ham) entrenado sobre el [2007 TREC Public Spam Corpus](https://plg.uwaterloo.ca/~gvcormac/treccorpus07/about.html), un conjunto de 75.419 correos reales etiquetados manualmente.

El pipeline cubre: parseo de correos en formato RFC822 (incluyendo multipart/MIME), limpieza de texto (HTML, URLs, stopwords, stemming), vectorización con bag-of-words y clasificación con regresión logística.

## 1. Carga del dataset

El índice del corpus (`full/index`) mapea cada correo con su etiqueta. La función `read_raw_emails` del módulo `spam_detector.dataset` se encarga de leerlo y devolver el texto crudo de cada mensaje.

In [1]:
import sys
sys.path.insert(0, "../src")

from spam_detector import read_raw_emails

DATASET_INDEX = "../data/trec07p/full/index"

raw_emails, labels = read_raw_emails(DATASET_INDEX, limit=6000)
print(f"Correos cargados: {len(raw_emails)}")
conteo = {label: labels.count(label) for label in set(labels)}
print(f"Distribución de clases: {conteo}")

Correos cargados: 6000
Distribución de clases: {'spam': 4488, 'ham': 1512}


In [2]:
print(raw_emails[0][:500])

From RickyAmes@aol.com  Sun Apr  8 13:07:32 2007
Return-Path: <RickyAmes@aol.com>
Received: from 129.97.78.23 ([211.202.101.74])
	by speedy.uwaterloo.ca (8.12.8/8.12.5) with SMTP id l38H7G0I003017;
	Sun, 8 Apr 2007 13:07:21 -0400
Received: from 0.144.152.6 by 211.202.101.74; Sun, 08 Apr 2007 19:04:48 +0100
Message-ID: <WYADCKPDFWWTWTXNFVUE@yahoo.com>
From: "Tomas Jacobs" <RickyAmes@aol.com>
Reply-To: "Tomas Jacobs" <RickyAmes@aol.com>
To: the00@speedy.uwaterloo.ca
Subject: Generic Cialis, brande


## 2. Parseo y limpieza de texto

`EmailParser` (en `spam_detector.email_parser`) aplica, en orden:

1. Extracción del cuerpo del mensaje (soporta correos multipart/MIME).
2. Eliminación de tags HTML.
3. Eliminación de URLs.
4. Tokenización.
5. Eliminación de stopwords y puntuación, aplicando *stemming* (Porter).

In [3]:
from spam_detector import EmailParser

parser = EmailParser()
print(parser.parse(raw_emails[0]))

do feel pressur perform rise occas tri viagra ..... anxieti thing past back old self


In [4]:
processed_emails = [parser.parse(email) for email in raw_emails]

## 3. Vectorización (bag-of-words)

Convertimos cada correo procesado en un vector de frecuencias de palabras con `CountVectorizer`.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(processed_emails)
print(f"Dimensiones de la matriz de features: {X.shape}")

Dimensiones de la matriz de features: (6000, 328896)


## 4. División train / test

Usamos un split estratificado para mantener la misma proporción de spam/ham en entrenamiento y test.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"Entrenamiento: {X_train.shape[0]} correos")
print(f"Test: {X_test.shape[0]} correos")

Entrenamiento: 4800 correos
Test: 1200 correos


## 5. Entrenamiento

Entrenamos un modelo de regresión logística, adecuado como baseline para clasificación de texto con features de tipo bag-of-words.

In [7]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## 6. Evaluación

Además del accuracy, revisamos precision, recall y F1 por clase, y la matriz de confusión para entender qué tipo de errores comete el modelo.

In [8]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.991

              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       302
        spam       0.99      1.00      0.99       898

    accuracy                           0.99      1200
   macro avg       0.99      0.99      0.99      1200
weighted avg       0.99      0.99      0.99      1200



In [9]:
import pandas as pd

labels_order = sorted(set(labels))
cm = confusion_matrix(y_test, y_pred, labels=labels_order)
pd.DataFrame(
    cm,
    index=[f"real: {l}" for l in labels_order],
    columns=[f"predicho: {l}" for l in labels_order],
)

,predicho: ham,predicho: spam
real: ham,295,7
real: spam,4,894


## 7. Probar con un correo nuevo

Ejemplo de cómo usar el pipeline completo (parseo + vectorización + modelo) para clasificar un correo nuevo.

In [10]:
def predecir(texto_crudo):
    texto_procesado = parser.parse(texto_crudo)
    vector = vectorizer.transform([texto_procesado])
    return clf.predict(vector)[0]

predecir(raw_emails[0])

np.str_('spam')

## Próximos pasos

- Probar `TfidfVectorizer` en lugar de `CountVectorizer` para ponderar mejor las palabras poco frecuentes.
- Comparar la regresión logística con `MultinomialNB` o `SVM`, algoritmos habituales en clasificación de texto.
- Entrenar sobre el corpus completo (75.419 correos) en lugar de una muestra de 6.000.
- Serializar el modelo y el vectorizador (`joblib`) para poder reutilizarlos sin reentrenar.